# Spatial Analysis of Emergency Medical Response Coverage in Mombasa

### The Problem
In dense urban environments, traffic congestion and suboptimal ambulance placement can severely delay emergency medical responses. The "golden hour" (the first 60 minutes following a traumatic injury or medical crisis) is critical in emergency medicine, making it essential to identify geographic "blind spots" that fall outside a standard 10-minute response window from major hospitals.

### Why I'm Doing This Project
This project applies spatial data science to a real-world logistics and resource allocation problem. It bridges the gap between academic data modeling and practical supply chain optimization, demonstrating how spatial analytics can directly improve operational efficiency and potentially save lives.

### Potential Outcome
The final output will be an interactive, data-driven map of Mombasa highlighting current ambulance coverage zones (isochrones) and pinpointing specific neighborhoods that are statistically underserved based on simulated emergency data.

### Methodology (The 7-Day Plan)
This project is being built iteratively using Python:
* **Mapping Engine:** `folium` for interactive leaflet rendering.
* **Routing API:** OpenRouteService (ORS) for calculating real-world driving times and generating spatial polygons.
* **Spatial Mathematics:** `shapely` for calculating intersections between simulated emergency coordinates and coverage zones.

### Lessons Learned & Skills Applied
*(I will update this section as the project progresses)*
* API integration and JSON parsing for spatial data.
* Transforming and rendering GeoJSON polygons.
* Applying spatial algorithms to quantify visual map data.

In [11]:
import folium

#creating map of Mombasa
mombasa_map = folium.Map(location=[-4.0435, 39.6682], zoom_start=14)

#storing major hospital coordinates in disctinoary
hospitals = {
    "Coast General Hospital": [-4.048805250648558, 39.673868806872676],
    "Aga Khan Hospital Mombasa": [-4.0698688656582735, 39.670492924060824],
    "Tudor Sub-County Hospital": [-4.040276959026293, 39.66393785289597],
    "Pandya Memorial Hospital": [-4.068095452652383, 39.67127391056656],
    "MEWA Hospital": [-4.053103663895366, 39.66146602406077],
    "Al-Farooq Hospital": [-4.048639046483364, 39.66540045289602],
    "Ganjoni Hospital": [-4.063481894045143, 39.65987433684035],
    "The Mombasa Hospital": [-4.064648766332206, 39.680340435708224]
}

#location pin for each hospital
for name, coords in hospitals.items():
    folium.Marker(
        location=coords,
        popup=name,
        icon=folium.Icon(color="red", icon="plus", prefix="fa")
    ).add_to(mombasa_map)

#display updated map
mombasa_map

In [12]:
import requests
import json

#Aopen Route Service API key
ORS_API_KEY = "eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjUxNjllZDJiNGIzNDQ1MzU4NDI1Yzk3MjU2M2FmM2QwIiwiaCI6Im11cm11cjY0In0="

#use specific ORS URL for car driving isochrones
url = "https://api.openrouteservice.org/v2/isochrones/driving-car"

#security headers to confirm with server
headers = {
    'Accept': 'application/json, application/geo+json, application/gpx+xml, img/png; charset=utf-8',
    'Authorization': ORS_API_KEY,
    'Content-Type': 'application/json; charset=utf-8'
}

body = {
    "locations": [[39.6705, -4.0531]], # Coast General (Longitude & Lattitude flipped)
    "range": [600],
    "range_type": "time"
}

#send request and save server's repl
response = requests.post(url, json=body, headers=headers)

#Checking
print(f"API Status Code: {response.status_code}")
if response.status_code == 200:
    print("\nSuccess! Here is a snippet of the raw GeoJSON data:")
    print(str(response.json())[:300] + "...")
else:
    print("Uh oh, check your API key or formatting.")

API Status Code: 200

Success! Here is a snippet of the raw GeoJSON data:
{'type': 'FeatureCollection', 'bbox': [39.614733, -4.082251, 39.708664, -3.999803], 'features': [{'type': 'Feature', 'properties': {'group_index': 0, 'value': 600.0, 'center': [39.67065164980744, -4.053080959842621]}, 'geometry': {'coordinates': [[[39.621017, -4.01135], [39.617204, -4.017881], [39.6...


In [13]:
import requests
import time

#dictionary to store our fetched polygon data
isochrones_data = {}

print("Fetching 10-minute driving zones from ORS API...\n")

#looping through all the hospitals from dictionary
for name, coords in hospitals.items():
    lon, lat = coords[1], coords[0]

    body = {
        "locations": [[lon, lat]],
        "range": [600],
        "range_type": "time"
    }

    #send request
    response = requests.post(url, json=body, headers=headers)

    #checking response and saving data
    if response.status_code == 200:
        #extracting JSON and save  hospital's name
        isochrones_data[name] = response.json()
        print(f"✅ Successfully fetched data for {name}")
    else:
        print(f"❌ Failed for {name}. Status code: {response.status_code}")

    #1 second delay
    time.sleep(1)

print("\nAll API calls complete! Data is ready to be mapped.")

Fetching 10-minute driving zones from ORS API...

✅ Successfully fetched data for Coast General Hospital
✅ Successfully fetched data for Aga Khan Hospital Mombasa
✅ Successfully fetched data for Tudor Sub-County Hospital
✅ Successfully fetched data for Pandya Memorial Hospital
✅ Successfully fetched data for MEWA Hospital
✅ Successfully fetched data for Al-Farooq Hospital
✅ Successfully fetched data for Ganjoni Hospital
✅ Successfully fetched data for The Mombasa Hospital

All API calls complete! Data is ready to be mapped.
